# RPTQ / QuIK / Atom：W4A4 的重排与缩放

**对应博客篇目**：PTQ 系列第 11 篇《RPTQ、QUIK 与 Atom：W4A4 的三块基石——重排、混合精度与融合 kernel》

---

## 原理概述

当激活也压到 4-bit（W4A4）时，per-group 量化面临的核心矛盾是：只要一个组里混进离群通道，整组的步长被劫持，组内所有正常通道陪葬。

三块递进的基石：
1. **RPTQ**：用通道重排把离群值隔离到同一组，其余组获得干净的细步长——数学上免费（置换矩阵 $XP(WP)^T = XW^T$），代价全在工程侧。
2. **QUIK**：端到端 4-bit 推理引擎，证明 W4A4 不只是论文设定。
3. **ATOM**：集大成——RPTQ 重排 + 双路混合精度（离群列 int8 + 主干 int4）+ 分组反量化累加 GEMM，把误差压到 W4 权重噪声地板的 1.02x。

本 demo 用 numpy 复现这条主线：**naive per-group A4 的离群值污染 → RPTQ 重排的修复 → ATOM 双路的贴地**。

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 固定随机种子，确保结果可复现（与 run.py 及博客正文一致）
SEED = 0
rng = np.random.default_rng(SEED)
print(f"随机种子: SEED={SEED}")

随机种子: SEED=0


## 第一步：构造 toy 数据

模拟 Transformer 线性层的输入激活 $X$ 和权重 $W$。正常通道取 $\mathcal{N}(0,1)$，随机挑出若干通道赋予 20-40 倍均值作为离群通道——这正是 LLM 激活中常见的「少数通道幅值远大于其余」的形态。

In [2]:
# ── 场景参数 ──
T      = 256   # token 序列长度
d_in   = 128   # 输入通道数（含离群通道）
d_out  = 128   # 输出通道数
n_out  = 8     # 离群通道数量
g      = 32    # per-group 量化组大小（d_in / g = 4 组）

# 正常通道：标准正态分布
X = rng.normal(0, 1.0, (T, d_in))

# 随机选出 n_out 个通道作为离群通道，赋予大均值 + 小波动
oc = sorted(rng.choice(d_in, n_out, replace=False).tolist())
mus = rng.uniform(20.0, 40.0, n_out)
for k, j in enumerate(oc):
    X[:, j] = mus[k] + 0.5 * rng.normal(0, 1.0, T)

# 权重：标准正态 × 0.02（模拟真实模型权重尺度）
W = rng.normal(0, 0.02, (d_out, d_in))

# 浮点参考输出
Y_ref = X @ W.T

print(f"X shape: {X.shape}   W shape: {W.shape}   Y_ref shape: {Y_ref.shape}")
print(f"离群通道索引: {oc}")
print(f"离群通道均值范围: [{min(mus):.1f}, {max(mus):.1f}]")
normal_idx = [i for i in range(d_in) if i not in oc]
print(f"正常通道 |X| max: {np.abs(X[:, normal_idx]).max():.4f}")

X shape: (256, 128)   W shape: (128, 128)   Y_ref shape: (256, 128)
离群通道索引: [4, 10, 30, 54, 71, 92, 107, 123]
离群通道均值范围: [20.7, 39.8]
正常通道 |X| max: 4.4941


## 第二步：量化工具函数

三个核心构件：

- `sym_quant`：对称 per-tensor 量化（将浮点值映射到 $[-(2^{b-1}-1),\,+(2^{b-1}-1)]$ 整数网格再还原）
- `group_act_quant`：逐组（per-token × per-group-of-channels）对称量化激活——每组独立计算 scale
- `rel_err`：Frobenius 范数相对误差，衡量量化输出与浮点参考的偏差

In [3]:
def sym_quant(A, bits):
    """对称 per-tensor 量化"""
    if A.size == 0:
        return A
    s = np.abs(A).max() / (2 ** (bits - 1) - 1)
    return np.round(A / s) * s


def group_act_quant(X, bits, g):
    """逐组(per-token x per-group-of-channels)对称量化激活"""
    T, d = X.shape
    Xq = np.empty_like(X)
    for j0 in range(0, d, g):
        blk = X[:, j0:j0 + g]
        s = np.abs(blk).max() / (2 ** (bits - 1) - 1)
        Xq[:, j0:j0 + g] = np.round(blk / s) * s
    return Xq


def rel_err(Y, Y_ref):
    """Frobenius 范数相对误差"""
    return float(np.linalg.norm(Y - Y_ref) / np.linalg.norm(Y_ref))


print("量化工具函数已定义: sym_quant, group_act_quant, rel_err")

量化工具函数已定义: sym_quant, group_act_quant, rel_err


## 第三步：权重 RTN-W4 地板

所有变体共享同一块权重（RTN per-channel 4-bit），只改变激活处理方式。权重本身的量化噪声构成误差地板——激活侧无论如何努力都无法穿越的下限。

In [4]:
# 权重统一 RTN per-channel 4-bit
Wq = np.stack([sym_quant(W[i], 4) for i in range(d_out)])
e_floor = rel_err(X @ Wq.T, Y_ref)
print(f"[W4 地板] 仅权重量化噪声（激活 fp16）：相对误差 = {e_floor:.4f}")

[W4 地板] 仅权重量化噪声（激活 fp16）：相对误差 = 0.1220


## 第四步：污染比分析

定义组内**污染比** $\rho_i = \max_{j \in G_i} |x_j| \ / \ \mathrm{median}_{j \in G_i} |x_j|$。$\rho \approx 1$ 表示组内通道幅值均匀、网格利用率高；$\rho \gg 1$ 表示组内混入离群值，正常通道的有效分辨率骤降。

重排的本质就是把 $\rho$ 大的组压缩到尽可能少的数量。

In [5]:
def contamination_ratio(X, g):
    """计算各组的污染比 rho = max/median of channel absmax"""
    _, d = X.shape
    rhos = []
    for j0 in range(0, d, g):
        blk = X[:, j0:j0 + g]
        col_absmax = np.abs(blk).max(axis=0)  # 各通道的 absmax
        rho = col_absmax.max() / np.median(col_absmax)
        rhos.append(round(float(rho), 2))
    return rhos


rho_before = contamination_ratio(X, g)
print(f"重排前各组污染比 rho: {rho_before}")
print("  -> 每个组都含有离群通道，所有组的步长都被劫持")
print()

# 先计算重排后的顺序（供后续步骤使用）
absmax_c = np.abs(X).max(0)
order = np.argsort(absmax_c)  # 从小到大排序 -> 离群通道聚到尾端
Xr = X[:, order]

rho_after = contamination_ratio(Xr, g)
print(f"重排后各组污染比 rho: {rho_after}")
print("  -> 前三组 rho~1（干净），离群值被隔离到最后一组")

重排前各组污染比 rho: [12.95, 10.11, 12.76, 12.28]
  -> 每个组都含有离群通道，所有组的步长都被劫持

重排后各组污染比 rho: [1.07, 1.04, 1.05, 11.1]
  -> 前三组 rho~1（干净），离群值被隔离到最后一组


## 第五步：Naive per-group A4

不做任何重排，直接按原始通道顺序分组量化。每个组的步长由组内最大幅值决定——如果组里混进了 40 倍离群值，正常通道的 15 级网格实际只剩约 1-2 有效级。

In [6]:
e0 = rel_err(group_act_quant(X, 4, g) @ Wq.T, Y_ref)
print(f"[Naive A4]   层输出相对误差 = {e0:.4f}  （为地板的 {e0/e_floor:.2f}x）")

[Naive A4]   层输出相对误差 = 0.1741  （为地板的 1.43x）


## 第六步：RPTQ 重排 + per-group A4

按通道 absmax 从小到大排序（原论文用 (min,max) 二维 k-means，一维排序是合理的简化）。重排后离群通道聚到尾端组，其余三个组拿到干净的细步长。

**置换等价性**：同时对激活列和权重列做相同重排，$(XP)(WP)^T = XPP^TW^T = XW^T$——浮点结果严格不变。

In [7]:
# 诊断：重排前各组的 absmax 上界
bounds_before = [round(float(np.abs(X[:, j0:j0+g]).max()), 2)
                 for j0 in range(0, d_in, g)]
print(f"重排前各组 absmax 上界: {bounds_before}")

# 重排激活和权重（权重同序是 RPTQ 工程三部曲之一）
Wrq = Wq[:, order]  # 权重按同一 order 重排

# 诊断：重排后各组的 absmax 上界
bounds_after = [round(float(np.abs(Xr[:, j0:j0+g]).max()), 2)
                for j0 in range(0, d_in, g)]
print(f"重排后各组 absmax 上界: {bounds_after}")
print("  -> 前三组上界骤降至 ~3-4，最后一组仍 ~40（隔离离群值）")
print()

# 重排后 per-group A4
Xq1 = group_act_quant(Xr, 4, g)
e1 = rel_err(Xq1 @ Wrq.T, Y_ref)
print(f"[RPTQ A4]    层输出相对误差 = {e1:.4f}  （为地板的 {e1/e_floor:.2f}x）")

重排前各组 absmax 上界: [41.09, 31.17, 39.08, 36.77]
重排后各组 absmax 上界: [2.86, 3.1, 3.35, 41.09]
  -> 前三组上界骤降至 ~3-4，最后一组仍 ~40（隔离离群值）

[RPTQ A4]    层输出相对误差 = 0.1417  （为地板的 1.16x）


## 第七步：ATOM 双路混合精度

离群通道走 int8 短支路（步长更细、精度更高），正常通道走 int4 组量化主干。两路结果相加即为最终输出。这是 LLM.int8() 分解思想在 W4A4 上的推广——分解开销从“整层”缩到“几列”。

In [8]:
# 构造正常/离群掩码
normal_mask = np.ones(d_in, dtype=bool)
normal_mask[oc] = False

# 主干：正常通道 int4 组量化
Xn_q = group_act_quant(X[:, normal_mask], 4, g)
# 短支路：离群通道 int8 对称量化
Xo_q = sym_quant(X[:, oc], 8)

# 双路结果相加
e2 = rel_err(Xn_q @ Wq[:, normal_mask].T + Xo_q @ Wq[:, oc].T, Y_ref)
print(f"[ATOM 双路]  层输出相对误差 = {e2:.4f}  （为地板的 {e2/e_floor:.2f}x）")

[ATOM 双路]  层输出相对误差 = 0.1236  （为地板的 1.01x）


## 第八步：置换等价性验证

验证重排在数学上是严格免费的：构造置换矩阵 $P$，计算 $\|XP(WP)^T - XW^T\|_{\max}$，预期偏差在浮点噪声级别（$\sim 10^{-14}$）。

In [9]:
P = np.eye(d_in)[order]
lhs = X @ W.T
rhs = (X @ P) @ (W @ P).T
perm_err = float(np.abs(rhs - lhs).max())
print(f"[置换等价性] ||XP(WP)^T - XW^T||_max = {perm_err:.2e}")
print("结论：偏差在浮点噪声级别，重排不改变任何数学结果")
print("      代价全在工程侧（LN 写回、权重同序、通道对齐）")

[置换等价性] ||XP(WP)^T - XW^T||_max = 7.99e-15


结论：偏差在浮点噪声级别，重排不改变任何数学结果
      代价全在工程侧（LN 写回、权重同序、通道对齐）


## 第九步：可视化

左图：重排前后各组 absmax 上界对比（对数刻度），直观展示 RPTQ 如何把离群值隔离到尾端组。
右图：三变体相对误差对比，展示从 naive → RPTQ → ATOM 的精度递进。

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 图 A：重排前后各组 absmax 上界 ──
ax = axes[0]
n_groups = d_in // g
xs = np.arange(n_groups)
ax.bar(xs - 0.2, bounds_before, width=0.4, color="#C44E52",
       label="重排前（自然布局）")
ax.bar(xs + 0.2, bounds_after, width=0.4, color="#4C72B0",
       label="RPTQ 重排后（absmax 排序）")
ax.set_yscale("log")
ax.set_xlabel(f"激活组索引（每组 {g} 通道）")
ax.set_ylabel("组内 absmax 上界（对数刻度）")
ax.set_title("RPTQ：排序让离群值聚到尾端组")
ax.grid(alpha=0.3, axis="y", which="both")
ax.legend(fontsize=9)

# ── 图 B：三变体相对误差 vs W4 地板 ──
ax = axes[1]
names = ["W4 地板\n(激活 fp16)", "Naive\nper-group A4",
         "RPTQ 重排\n+ A4", "ATOM 双路\n(离群 int8 + int4)"]
vals = [e_floor, e0, e1, e2]
colors = ["#999999", "#C44E52", "#DD8452", "#55A868"]
bars = ax.bar(names, vals, color=colors)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v * 1.03,
            f"{v:.4f}\n({v/e_floor:.2f}x)", ha="center", fontsize=8.5)
ax.set_ylabel("层输出相对误差")
ax.set_title("同一块 W4 权重，不同激活处理的误差对比")
ax.grid(alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20333/499296026.py:32: UserWarning: Glyph 28608 (\N{CJK UNIFIED IDEOGRAPH-6FC0}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20333/499296026.py:32: UserWarning: Glyph 27963 (\N{CJK UNIFIED IDEOGRAPH-6D3B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20333/499296026.py:32: UserWarning: Glyph 32452 (\N{CJK UNIFIED IDEOGRAPH-7EC4}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20333/499296026.py:32: UserWarning: Glyph 32034 (\N{CJK UNIFIED IDEOGRAPH-7D22}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_20333/499296026.py:32: UserWarning: Glyph 24341 (\N{CJK UNIFIED IDEOGRAPH-5F15}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/1h/x16wsv8n5

## 结果解读

### 关键数字汇总

| 变体 | 相对误差 | 为地板的倍数 | 说明 |
|:---|:---:|:---:|:---|
| W4 权重地板 | 0.1220 | 1.00x | 激活 fp16，仅权重 RTN-W4 噪声 |
| Naive per-group A4 | 0.1741 | 1.43x | 每个组都含离群通道，步长全部被劫持 |
| RPTQ 重排 + A4 | 0.1417 | 1.16x | 离群值隔离到尾端组，三个干净组拿到细步长 |
| ATOM 双路 | 0.1236 | 1.01x | 离群列 int8 + 主干 int4，误差贴住地板 |

> 注：以上为 SEED=0、T=256、d=128、g=32 的实测结果。博客正文用 T=2048、d_out=256，数值略有差异但量级一致（地板 0.1153、naive 1.51x、RPTQ 1.20x、ATOM 1.02x）。

### 核心观察

1. **污染比是划分 $\mathcal{G}$ 的函数**：同一组数据，换一种分法，每个组的 $\rho$ 都可以不同。RPTQ 的贡献就是系统性地利用了这个自由度。
2. **重排是“隔离”而非“治疗”**：离群组自身依旧粗糙（步长 $\sim 40/7 \approx 5.8$ 对 $\pm 0.5$ 的波动毫无分辨率），但其余组获得干净步长，总误差显著下降。
3. **ATOM 双路把激活侧的账基本还清**：剩余误差几乎全部来自权重 RTN 本身。W4A4 的瓶颈回到权重精度。

### 方法局限

1. **静态重排的脆弱性**：`order` 从校准集冻结，域外输入的离群格局可能漂移——虽然离群通道位置相当稳定，但边界情形会翻转。
2. **重排的连带税**：KV cache、attention 内部状态都要跟随同一 `order`，任何一处遗漏都是静默错误；多卡切分时还要保证各 rank 的 order 一致。
3. **双路方案的位宽账**：ATOM 给离群列发 int8，严格平均位宽高于标称的“4-bit”；对比压缩率时需要诚实还原。
4. **评估盲区**：W4A4 的对比大多停在 PPL + 少数任务，长上下文、代码、数学等对离群更敏感的场景覆盖不足。

### 参考文献

- RPTQ: arXiv:2304.01089
- QUIK: arXiv:2310.09259
- ATOM: arXiv:2310.19102